In [ ]:
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "scripts" / "paper_renamer.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Start this notebook from inside the THESIS repository.")
CROMA_DIR = REPO_ROOT / "CROMA" / "CROMA"
if not (CROMA_DIR / "use_croma.py").is_file():
    raise FileNotFoundError(f"CROMA checkout missing: {CROMA_DIR}")
sys.path.insert(0, str(CROMA_DIR))
from use_croma import PretrainedCROMA
print(PretrainedCROMA)


In [ ]:
from use_croma import PretrainedCROMA
print(PretrainedCROMA)

In [ ]:
import inspect

print(inspect.signature(PretrainedCROMA))

In [ ]:
print(inspect.getsource(PretrainedCROMA.__init__))

In [ ]:
import use_croma

print(use_croma.__file__)

In [ ]:
from huggingface_hub import hf_hub_download

checkpoint_path  = hf_hub_download(
    repo_id="antofuller/CROMA",
    filename="CROMA_base.pt",
    local_dir=str(CROMA_DIR)
)

print(checkpoint_path )

In [ ]:
import torch

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu"
)

print(type(checkpoint))
print(checkpoint.keys())

In [ ]:
s1_state = checkpoint["s1_encoder"]

print(type(s1_state))
print("Number of entries:", len(s1_state))

In [ ]:
for name, tensor in list(s1_state.items())[:20]:
    print(f"{name:60s} {tuple(tensor.shape)}")

In [ ]:
name, tensor = next(iter(s1_state.items()))

print("Parameter name:", name)
print("Shape:", tensor.shape)
print("dtype:", tensor.dtype)
print(tensor.flatten()[:10])

In [ ]:
for key, value in checkpoint.items():
    if isinstance(value, dict):
        n_params = sum(t.numel() for t in value.values() if torch.is_tensor(t))
        print(f"{key:25s} {n_params:,} parameters")
    else:
        print(key, type(value))

In [ ]:
model = PretrainedCROMA(
    pretrained_path=checkpoint_path,
    size="base",
    modality="both",
    image_resolution=120
)

In [ ]:
print(type(model))

In [ ]:
print(model)


In [ ]:
'''import torch
from torchview import draw_graph
import os
import shutil
import subprocess

graphviz_bin = r"C:\Program Files\Graphviz\bin"

os.environ["PATH"] += os.pathsep + graphviz_bin

print(shutil.which("dot"))


subprocess.run(["dot", "-V"])
sar = torch.randn(1, 2, 120, 120)

graph = draw_graph(
    model.s1_encoder,
    input_data=(sar, model.attn_bias),
    expand_nested=True,
    depth=4,
    graph_name="CROMA_SAR_Encoder"
)
'''
#graph.visual_graph

In [ ]:
print("PARAMETERS")

for name, parameter in list(model.named_parameters())[:10]:
    print(name, parameter.shape)

In [ ]:
print("BUFFERS")

for name, buffer in model.named_buffers():
    print(name, buffer.shape)

In [ ]:
print("STATE DICT")

for name, tensor in list(model.state_dict().items())[:15]:
    print(name, tensor.shape)

In [ ]:
p = model.s1_encoder.linear_input.weight

print(type(p))
print(p.requires_grad)

In [ ]:
print(type(model.attn_bias))
print(model.attn_bias.shape)

In [ ]:
print(
    any(name == "attn_bias"
        for name, _ in model.named_buffers())
)

print(
    "attn_bias" in model.state_dict()
)

In [ ]:
for name, tensor in list(model.state_dict().items())[:15]:
    print(name, tensor.shape)

In [ ]:
parameter_names = {
    name for name, _ in model.named_parameters()
}

state_names = set(model.state_dict().keys())

print("Parameters:", len(parameter_names))
print("State entries:", len(state_names))

print("State but not parameters:")
print(state_names - parameter_names)

In [ ]:
a = model.s1_encoder.linear_input.weight
b = model.state_dict()["s1_encoder.linear_input.weight"]

print(type(a))
print(type(b))